In [12]:
import pandas as pd
import jax.numpy as jnp
import jax.random as jr
import sys
import os

# Add parent directory to Python path
sys.path.append(os.path.dirname(os.getcwd()))

# Import from parent directory to handle relative imports
sys.path.append(os.getcwd())  # Add current directory too
from data import data_utils_integrative_model
import data_utils

In [13]:
config = {
        # General settings
        "seed": 0,  # random seed

        # data selection settings
        "data_kwargs": {
            "sampling_area": "North_South",
            "project": "both", # one of ESI_CorA, AMELAG
            "max_precipitation_subsetting": None, # one of None, dry, light_rain
            "substance_normalization": "flow", # one of None, PMMoV, flow
            "gene_target": "N1", # one of N1, N2
            "log_scale": True, # this only considers WW measurements, not case counts
        },
}


In [ ]:
y_name, date, t_scale, ts, ys, population_size, _ = data_utils_integrative_model.load_data(config["data_kwargs"])
# calculation of 7D sum means that we need I_new predictions for all days, not only observed ones
t_0, t_end = round(ts[0]*t_scale), round(ts[-1]*t_scale)
# we start 1 day earlier to calculate # new infections
t_all = jnp.arange(t_0-1, t_end + 1, dtype=jnp.float32)
t_mask = jnp.isin(t_all, ts*t_scale)[1:]
t_mask_ids = jnp.where(t_mask)[0]
assert t_mask.sum() == len(ts), "t_mask does not match the length of ts"
t_all = t_all/t_scale

In [15]:
population_size

176187

In [21]:
daily_new_infections = ys[:,1]/7

In [ ]:
E_0 = daily_new_infections[1]*1/0.5  # alpha = 0.5 # 1.7 – 2.5 days latent incubation period, 
I_0 = daily_new_infections[1]*3  # gamma = 0.33*self.t_scale # 3 - 3.5 days infection period
R_0 = 0.92*population_size # based on https://www.rki.de/DE/Themen/Infektionskrankheiten/Infektionskrankheiten-A-Z/C/COVID-19-Pandemie/AK-Studien/Ergebnisse.html

In [34]:
E_0

np.float64(862.8571428571429)

In [35]:
I_0

np.float64(1294.2857142857142)

In [36]:
R_0

162092.04

In [ ]:
862.857